# Silver Incremental — Orders (SCD1 via CDF + MERGE)
**GlobalMart | Tredence DE Advanced Training | Day 12 Pattern**

| | |
|---|---|
| **Source** | `gbmart.bronze.orders` — CDF enabled (Lakeflow Connect CDC) |
| **Target** | `gbmart.silver.orders` |
| **SCD Type** | SCD1 — orders are transactional facts; the latest state overwrites the previous |

### Why SCD1 for orders?
An order has **one current truth** — its current status, current delivery date. When
Lakeflow CDC tells us `OR-000478` moved from `Shipped` to `Delivered`, we overwrite
the row, not keep both versions. That is SCD1.

### The flow
| Step | What it does |
|---|---|
| 1 | Baseline — row counts before the run |
| 2 | Inspect Bronze history — identify the version to read from |
| 3 | CDF read — only the changed rows, not the full 126k+ table |
| 4 | Apply the same transforms as the full-load notebook |
| 5 | SCD1 MERGE into silver.orders |
| 6 | Verify — confirm the update + new inserts landed correctly |

## Step 1 — Setup + Baseline

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import DateType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

BRONZE_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.orders"
SILVER_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders"
VALID_CHANNELS = ["Online", "Retail PoS"]

baseline_count = spark.table(SILVER_TABLE).count()
print(f"silver.orders before this run : {baseline_count:,}")

## Step 2 — Inspect Bronze History
Find the version the initial full-load notebook (`02_orders_data_cleaning_dq_checks.ipynb`)
already consumed. Set `LAST_PROCESSED_VERSION` to that version number — CDF reads
everything **strictly after** it, which is only the Lakeflow-delivered changes.

In [ ]:
# DESCRIBE HISTORY on bronze.orders/order_items fails here with
# STREAMING_TABLE_OPERATION_NOT_ALLOWED.REQUIRES_SHARED_COMPUTE -- these Bronze
# tables are Streaming Tables (Lakeflow-managed), and DESCRIBE HISTORY on a
# Streaming Table requires a Shared cluster or a SQL warehouse, not the
# Assigned/No-Isolation cluster this notebook runs on. table_changes() is not
# subject to that restriction (cells below already use it successfully), so we
# rebuild the same per-version summary from it instead of DESCRIBE HISTORY.
# If you switch this notebook to a Shared cluster or SQL warehouse, the original
# spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}")...display() also works fine.
(
    spark.sql(f"""
        SELECT _commit_version, _commit_timestamp, _change_type, COUNT(*) AS row_count
        FROM table_changes('{BRONZE_TABLE}', 0)
        GROUP BY _commit_version, _commit_timestamp, _change_type
        ORDER BY _commit_version, _change_type
    """)
    .display()
)

In [ ]:
%sql
-- Version 0 (using table_changes, not VERSION AS OF -- see note below)
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.orders',
  0,
  0
);

In [ ]:
%sql
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.orders',
  0,
  10
);

In [ ]:
%sql
-- Version 1
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.orders',
  1
);

In [ ]:
%sql
-- Version 2
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.orders',
  2
);

In [ ]:
%sql
-- Version 3
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.orders',
  3
);

In [ ]:
%sql
-- Version 4
SELECT *
FROM table_changes(
  'harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.orders',
  4
);

In [ ]:
# Set to the version number of the initial full load (already consumed by silver full-load notebook)
LAST_PROCESSED_VERSION = 3   # <-- update from history output above

## Step 3 — CDF Read: Only the Changes, One Row Per Order
`readChangeFeed` + `startingVersion` returns every change event after
`LAST_PROCESSED_VERSION` -- that includes `insert`, `update_preimage` (the
*before* snapshot of an update), `update_postimage` (the *after* snapshot), and
`delete`. We only ever want the final state of a row, so we filter to `insert`
and `update_postimage`.

That filter alone isn't enough, though: if the *same* `order_id` changed more
than once inside this one incremental batch (two Lakeflow commits landed since
the last run), the filter still leaves two `update_postimage` rows for that
order_id -- and MERGE can't match two source rows to the same target row
(`DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE`). So we also keep only
the *latest* event per `order_id`, ranked by `_commit_version` -- Delta's own
monotonically-increasing commit counter for this table, which is a more
reliable tiebreaker than a source column like `updated_at` (no clock-skew or
equal-timestamp risk).

For the Day 12 scenario:
- `OR-000001`, `OR-000002`, `OR-000003` — Lakeflow CDC UPDATEs (`ActualDeliveryDate` filled)
- `OR-900001`, `OR-900002` — new inserts from Postgres via Lakeflow

In [ ]:
# Only keep the final state of each change -- insert and update_postimage.
# update_preimage (the "before" row of an update) and delete are dropped; this
# notebook's SCD1 MERGE only ever inserts or overwrites, it never deletes.
cdf_df_raw = (
    spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingVersion", LAST_PROCESSED_VERSION + 1)
        .table(BRONZE_TABLE)
        .filter(col("_change_type").isin("insert", "update_postimage"))
)

# De-duplicate to exactly one row per order_id -- keep the latest event, ranked
# by _commit_version (Delta's own monotonic commit counter, not a source column,
# so there is no clock-skew or hardcoded-version risk). This is what makes the
# MERGE below safe even if the same order_id changed more than once in one batch.
order_window = Window.partitionBy("orderid").orderBy(col("_commit_version").desc())

cdf_df = (
    cdf_df_raw
    .withColumn("_rn", row_number().over(order_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)

raw_count = cdf_df_raw.count()
deduped_count = cdf_df.count()
print(f"Changed rows via CDF (insert/update_postimage only) : {raw_count:,}")
print(f"Changed rows after de-duplicating to latest per order_id : {deduped_count:,}")
if raw_count != deduped_count:
    print(f"  -> {raw_count - deduped_count} duplicate order_id event(s) collapsed to their latest version")

cdf_df.select("orderid", "actualdeliverydate", "_change_type", "_commit_version").display()

## Step 4 — Apply Same Transforms as Full-Load Notebook
Identical logic to `02_orders_data_cleaning_dq_checks.ipynb` Step 9 —
date casts, derived KPI columns, `_data_note` for timezone-artifact rows.
Scoped to the small CDF result instead of all 126k+ rows.

In [ ]:
transformed_df = (
    cdf_df
    .withColumn("orderdate",            col("orderdate").cast(DateType()))
    .withColumn("shippingdate",         col("shippingdate").cast(DateType()))
    .withColumn("expecteddeliverydate", col("expecteddeliverydate").cast(DateType()))
    .withColumn("actualdeliverydate",   col("actualdeliverydate").cast(DateType()))
    .withColumn("order_to_ship_days",
        when(col("shippingdate").isNotNull(),
             datediff(col("shippingdate"), col("orderdate"))
        ).otherwise(lit(None).cast("int"))
    )
    .withColumn("ship_to_delivery_days",
        when(col("actualdeliverydate").isNotNull() & col("shippingdate").isNotNull(),
             datediff(col("actualdeliverydate"), col("shippingdate"))
        ).otherwise(lit(None).cast("int"))
    )
    .withColumn("delivery_delay_days",
        when(col("actualdeliverydate").isNotNull() & col("expecteddeliverydate").isNotNull(),
             datediff(col("actualdeliverydate"), col("expecteddeliverydate"))
        ).otherwise(lit(None).cast("int"))
    )
    .withColumn("is_delivered", col("actualdeliverydate").isNotNull())
    .withColumn("is_late",
        when(
            col("actualdeliverydate").isNotNull() & col("expecteddeliverydate").isNotNull(),
            col("actualdeliverydate") > col("expecteddeliverydate")
        ).otherwise(lit(None).cast("boolean"))
    )
    .withColumn("order_status",
        when(col("actualdeliverydate").isNotNull(), lit("Delivered"))
        .when(col("shippingdate").isNotNull(),      lit("Shipped"))
        .otherwise(                                  lit("Pending"))
    )
    .withColumn("_data_note",
        when(
            col("actualdeliverydate").isNotNull() & col("shippingdate").isNotNull() &
            (col("actualdeliverydate") < col("shippingdate")),
            lit("POSSIBLE_TIMEZONE_OFFSET_1DAY")
        ).otherwise(lit(None).cast("string"))
    )
    .withColumnRenamed("orderid",              "order_id")
    .withColumnRenamed("customerid",           "customer_id")
    .withColumnRenamed("orderdate",            "order_date")
    .withColumnRenamed("shippingdate",         "shipping_date")
    .withColumnRenamed("expecteddeliverydate", "expected_delivery_date")
    .withColumnRenamed("actualdeliverydate",   "actual_delivery_date")
    .withColumnRenamed("shippingtierid",       "shipping_tier_id")
    .withColumnRenamed("supplierid",           "supplier_id")
    .withColumnRenamed("orderchannel",         "order_channel")
    .withColumn("_silver_updated_at", current_timestamp())
    .select(
        "order_id", "customer_id", "order_date", "order_channel", "order_status",
        "shipping_tier_id", "supplier_id",
        "shipping_date", "expected_delivery_date", "actual_delivery_date",
        "order_to_ship_days", "ship_to_delivery_days", "delivery_delay_days",
        "is_delivered", "is_late", "_data_note", "updated_at", "_silver_updated_at"
    )
)

print(f"Rows to merge: {transformed_df.count():,}")
transformed_df.select("order_id", "order_status", "actual_delivery_date", "_change_type" if "_change_type" in transformed_df.columns else lit(None)).display()

## Step 5 — SCD1 MERGE into silver.orders

- **Matched** `order_id` → update all columns (overwrite — SCD1, no history kept)
- **Not matched** → insert as a new row (new order placed since last run)

This is safe to re-run. Running it twice on the same CDF batch produces the same
result — the second run matches and overwrites with identical data.

In [ ]:
silver_table = DeltaTable.forName(spark, SILVER_TABLE)

(silver_table.alias("tgt")
    .merge(transformed_df.alias("src"), "tgt.order_id = src.order_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print(f"MERGE complete")
print(f"silver.orders after this run: {spark.table(SILVER_TABLE).count():,}")

### Validation — Merge Metrics
Pulled directly from Delta's own transaction log for this MERGE, not
re-derived by hand -- this is the authoritative record of what the MERGE did.

In [ ]:
merge_metrics_row = (
    spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}")
    .orderBy(desc("version"))
    .limit(1)
    .select("version", "operation", "operationMetrics")
    .collect()[0]
)

print(f"MERGE metrics (silver.orders Delta version {merge_metrics_row['version']}):")
for k, v in merge_metrics_row["operationMetrics"].items():
    if k.startswith("numTarget") or k.startswith("numSource"):
        print(f"  {k:35s}: {v}")

In [ ]:
# Validation — duplicate order_ids. This is the exact condition that used to
# crash the MERGE with DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW_IN_MERGE.
# After the Step 3 fix above, this should always be empty -- if it ever isn't,
# stop here rather than let the MERGE fail (or worse, silently write bad data).
dupes = transformed_df.groupBy("order_id").count().filter("count > 1")
dupe_count = dupes.count()
print(f"Duplicate order_ids in transformed_df : {dupe_count}  (expected 0)")
if dupe_count > 0:
    dupes.display()
    raise AssertionError(
        f"Found {dupe_count} duplicate order_id(s) in transformed_df -- "
        "the Step 3 de-duplication should have prevented this. Do not proceed "
        "with the MERGE until this is fixed."
    )

In [ ]:
transformed_df \
    .filter(col("order_id").isin("OR-000001","OR-000002","OR-000003")) \
    .orderBy("order_id") \
    .display()

In [ ]:
transformed_df \
    .filter(col("order_id") == "OR-900001") \
    .display()

### Validation — Every Order in This Batch Landed in silver.orders
Generic check using the actual `cdf_df` order_id list from this run (not
hardcoded IDs) -- passes on any future incremental batch, not just this demo.

In [ ]:
batch_order_ids = [r["orderid"] for r in cdf_df.select("orderid").distinct().collect()]
landed = (
    spark.table(SILVER_TABLE)
    .filter(col("order_id").isin(batch_order_ids))
    .select("order_id")
    .distinct()
    .count()
)
print(f"Orders in this batch : {len(batch_order_ids)}")
print(f"Of those, now present in silver.orders : {landed}  (expected {len(batch_order_ids)})")
if landed != len(batch_order_ids):
    missing = set(batch_order_ids) - {
        r["order_id"] for r in spark.table(SILVER_TABLE).filter(col("order_id").isin(batch_order_ids)).select("order_id").collect()
    }
    raise AssertionError(f"{len(missing)} order_id(s) from this batch never landed in silver.orders: {missing}")

## Step 6 — Verify

Three things to confirm:
1. `OR-000001`, `OR-000002`, `OR-000003` now show `order_status = Delivered` with `actual_delivery_date` populated
2. `OR-900001` and `OR-900002` exist as new rows with `order_status = Pending`
3. Row count = baseline + 2 (the 2 new orders)

In [ ]:
df = spark.table(SILVER_TABLE)
print(f"silver.orders row count : {df.count():,}  (was {baseline_count:,} before run)")

# Check the 3 updated orders — ActualDeliveryDate should now be filled, status = Delivered
print("\n--- OR-000001 / OR-000002 / OR-000003 (should be Delivered) ---")
df.filter(col("order_id").isin(["OR-000001", "OR-000002", "OR-000003"])) \
  .select("order_id", "order_status", "actual_delivery_date", "delivery_delay_days", "_silver_updated_at") \
  .display()

# Check the 2 new orders
print("\n--- OR-900001 / OR-900002 (should exist as new rows, status = Pending) ---")
df.filter(col("order_id").isin(["OR-900001", "OR-900002"])) \
  .select("order_id", "customer_id", "order_date", "order_channel", "order_status") \
  .display()

In [ ]:
# Delta history — should show a new MERGE version
spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}") \
    .select("version", "timestamp", "operation", "operationMetrics") \
    .orderBy(desc("version")).limit(3).display()

## Reset (if needed)

In [ ]:
# Undo only the 2 new inserts. OR-000478 update is harder to roll back — use Time Travel if needed.
# spark.sql("DELETE FROM gbmart.silver.orders WHERE order_id IN ('OR-900001','OR-900002')")
# print("New orders removed — OR-000478 still shows updated state; use Time Travel to restore if needed")